In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

# ─── CONSTANTS ───
SQM_PER_DAY_SHIFT = 10_000 / 8       # 1,250 m²/day per shift
OPERATING_DAYS = 312                   # 6 days/week × 52 weeks
SQM_PER_ELEMENT = 54                   # m² of sheet consumed per element
COMMITTED_SHEET_M2 = 172_000           # existing customer commitment
ELEMENT_VALUE_MULT = 1.83             # elements sell at 1.83× the sheet-m² rate

CAPACITY_ONE_SHIFT = SQM_PER_DAY_SHIFT * OPERATING_DAYS  # 390,000 m²/yr

# ─── INPUT WIDGETS ───
_style = {'description_width': '160px'}
_layout = widgets.Layout(width='340px')

w_target = widgets.FloatText(value=5_000_000, description='Target Revenue ($):', step=100_000, style=_style, layout=_layout)
w_cost   = widgets.FloatText(value=9.50, description='Production Cost ($/m²):', step=0.10, style=_style, layout=_layout)
w_price  = widgets.FloatText(value=25.00, description='Sheet Mkt Price ($/m²):', step=0.50, style=_style, layout=_layout)

w_go  = widgets.Button(description='▶  Analyze Plan', button_style='primary',
                       layout=widgets.Layout(width='220px', height='38px'))
w_out = widgets.Output()


# ─── CORE SOLVER ───
def solve_plan(target, cost, price, shifts):
    """Find the mix that hits target revenue with max margin for a given shift count.

    Strategy: elements have higher margin per m² than sheet (1.83× revenue per m²
    of material vs 1×, at the same production cost), so prefer elements first,
    then fill any remaining revenue gap with additional sheet.
    """
    import math
    cap = CAPACITY_ONE_SHIFT * shifts
    elem_unit_price = ELEMENT_VALUE_MULT * price * SQM_PER_ELEMENT

    if cap < COMMITTED_SHEET_M2:
        return None

    avail_m2 = cap - COMMITTED_SHEET_M2          # m² available after commitment
    committed_rev = COMMITTED_SHEET_M2 * price
    remaining_rev = target - committed_rev

    if remaining_rev <= 0:
        # Committed sheet alone meets/exceeds target — produce only commitment
        n_elem = 0
        extra_sheet = 0
    else:
        # Max elements that fit in available capacity
        max_elem = int(avail_m2 // SQM_PER_ELEMENT)
        # Elements needed to cover remaining revenue (prefer elements for margin)
        elem_needed = math.ceil(remaining_rev / elem_unit_price)
        n_elem = min(elem_needed, max_elem)

        elem_rev_so_far = n_elem * elem_unit_price
        gap = remaining_rev - elem_rev_so_far
        if gap > 0 and price > 0:
            # Fill gap with additional sheet
            sheet_m2_needed = math.ceil(gap / price)
            avail_after_elem = avail_m2 - n_elem * SQM_PER_ELEMENT
            extra_sheet = min(sheet_m2_needed, int(avail_after_elem))
        else:
            extra_sheet = 0

    total_sheet  = COMMITTED_SHEET_M2 + extra_sheet
    total_m2     = total_sheet + n_elem * SQM_PER_ELEMENT
    sheet_rev    = total_sheet * price
    elem_rev     = n_elem * elem_unit_price
    revenue      = sheet_rev + elem_rev
    prod_cost    = total_m2 * cost
    margin       = revenue - prod_cost

    # Per-shift breakdown
    s1_used = min(total_m2, CAPACITY_ONE_SHIFT)
    s1_pct  = s1_used / CAPACITY_ONE_SHIFT * 100
    s2_used = max(0, total_m2 - CAPACITY_ONE_SHIFT) if shifts == 2 else 0
    s2_pct  = (s2_used / CAPACITY_ONE_SHIFT * 100) if shifts == 2 else 0

    return dict(
        shifts=shifts, capacity=cap,
        elements=n_elem, elem_price=elem_unit_price,
        extra_sheet=extra_sheet, total_sheet=total_sheet,
        total_m2=total_m2, utilization=total_m2 / cap * 100,
        revenue=revenue, cost=prod_cost,
        margin=margin, margin_pct=(margin / revenue * 100) if revenue else 0,
        sheet_rev=sheet_rev, elem_rev=elem_rev,
        s1_used=s1_used, s1_pct=s1_pct,
        s2_used=s2_used, s2_pct=s2_pct,
    )


# ─── CLICK HANDLER ───
def on_analyze(_):
    target = w_target.value
    cost   = w_cost.value
    price  = w_price.value

    with w_out:
        clear_output(wait=True)

        if price <= 0 or cost <= 0 or target <= 0:
            display(HTML("<h3 style='color:#c0392b'>All inputs must be positive.</h3>"))
            return

        p1 = solve_plan(target, cost, price, 1)
        p2 = solve_plan(target, cost, price, 2)

        # ── Determine recommendation ──
        if p1 and p1['revenue'] >= target:
            rec, label = p1, '✓ Feasible with 1 Shift'
        elif p2 and p2['revenue'] >= target:
            rec, label = p2, '✓ Feasible — 2 Shifts Required'
        else:
            max_rev = p2['revenue'] if p2 else (p1['revenue'] if p1 else 0)
            rec   = p2 if p2 else p1
            label = f'✗ NOT Feasible (${target - max_rev:,.0f} short of target even at 2 shifts)'

        feasible = rec['revenue'] >= target

        # ── Helper for table rows ──
        def row(lbl, val, bold=False, bg=''):
            b = 'font-weight:bold;' if bold else ''
            s = f'background:{bg};' if bg else ''
            return (f'<tr style="{s}"><td style="padding:6px 10px;border:1px solid #eee;{b}">{lbl}</td>'
                    f'<td style="padding:6px 10px;border:1px solid #eee;text-align:right;{b}">{val}</td></tr>')

        elem_m2 = rec['elements'] * SQM_PER_ELEMENT

        # Build shift 2 rows only when needed
        shift2_rows = ''
        if rec['shifts'] == 2:
            shift2_rows = (
                row('Shift 2 Capacity', f'{CAPACITY_ONE_SHIFT:,.0f} m²')
                + row('Shift 2 Used', f'{rec["s2_used"]:,.0f} m² ({rec["s2_pct"]:.1f}%)',
                      bold=True, bg='#eaf7ea' if rec['s2_pct'] <= 100 else '#fde8e8')
            )

        html = (
            f'<div style="font-family:system-ui,sans-serif;">'
            f'<h2 style="color:{"#27ae60" if feasible else "#c0392b"};margin:0 0 8px;">{label}</h2>'
            + (('<p style="color:#c0392b"><b>Increase market price or lower target.</b></p>') if not feasible else '')
            + f'''
        <table style="border-collapse:collapse;width:100%;margin:10px 0;">
        <tr style="background:#f5f5f5"><th colspan="2" style="text-align:left;padding:8px;border:1px solid #ddd;">Production Plan</th></tr>
        {row('Committed Sheet',             f'{COMMITTED_SHEET_M2:,.0f} m²')}
        {row('Additional Sheet Sold',       f'{rec["extra_sheet"]:,.0f} m²')}
        {row('Total Sheet Sold',            f'{rec["total_sheet"]:,.0f} m²', bold=True)}
        {row('Elements Produced',           f'{rec["elements"]:,.0f} units')}
        {row('Sheet Consumed for Elements', f'{elem_m2:,.0f} m²')}
        {row('Total m² Produced',           f'{rec["total_m2"]:,.0f} m²', bold=True)}
        </table>

        <table style="border-collapse:collapse;width:100%;margin:10px 0;">
        <tr style="background:#f5f5f5"><th colspan="2" style="text-align:left;padding:8px;border:1px solid #ddd;">Pricing</th></tr>
        {row('Sheet Selling Price',   f'${price:,.2f} /m²')}
        {row('Element Selling Price', f'${rec["elem_price"]:,.2f} /unit')}
        {row('<span style="color:#888;font-size:0.85em;">= 1.83 &times; $' + f'{price:,.2f}/m² &times; 54 m²</span>', '')}
        </table>

        <table style="border-collapse:collapse;width:100%;margin:10px 0;">
        <tr style="background:#f5f5f5"><th colspan="2" style="text-align:left;padding:8px;border:1px solid #ddd;">Financials</th></tr>
        {row('Sheet Revenue',    f'${rec["sheet_rev"]:,.0f}')}
        {row('Element Revenue',  f'${rec["elem_rev"]:,.0f}')}
        {row('Total Revenue',    f'${rec["revenue"]:,.0f}', bold=True, bg='#eaf7ea' if feasible else '#fde8e8')}
        {row('Revenue Target',   f'${target:,.0f}')}
        {row('Production Cost',  f'${rec["cost"]:,.0f}')}
        {row('Gross Margin',     f'${rec["margin"]:,.0f}  ({rec["margin_pct"]:.1f}%)', bold=True,
             bg='#eaf7ea' if rec["margin"] > 0 else '#fde8e8')}
        </table>

        <table style="border-collapse:collapse;width:100%;margin:10px 0;">
        <tr style="background:#f5f5f5"><th colspan="2" style="text-align:left;padding:8px;border:1px solid #ddd;">Capacity &amp; Shifts</th></tr>
        {row('Shifts Required',       str(rec['shifts']))}
        {row('Total Annual Capacity', f'{rec["capacity"]:,.0f} m²')}
        {row('Total Utilization',     f'{rec["utilization"]:.1f}%', bold=True)}
        {row('Shift 1 Capacity',      f'{CAPACITY_ONE_SHIFT:,.0f} m²')}
        {row('Shift 1 Used',          f'{rec["s1_used"]:,.0f} m² ({rec["s1_pct"]:.1f}%)',
             bold=True, bg='#eaf7ea' if rec['s1_pct'] <= 100 else '#fde8e8')}
        {shift2_rows}
        </table>
        </div>'''
        )
        display(HTML(html))

        # ── Charts ──
        if HAS_PLOTLY:
            fig = make_subplots(rows=1, cols=2,
                specs=[[{'type': 'indicator'}, {'type': 'bar'}]],
                column_widths=[0.4, 0.6])

            gauge_max = max(rec['revenue'], target) * 1.2
            fig.add_trace(go.Indicator(
                mode='gauge+number+delta',
                value=rec['revenue'], delta={'reference': target, 'valueformat': '$,.0f'},
                number={'valueformat': '$,.0f'}, title={'text': 'Revenue vs Target'},
                gauge={
                    'axis': {'range': [0, gauge_max]},
                    'bar': {'color': '#27ae60' if feasible else '#e74c3c'},
                    'steps': [{'range': [0, target], 'color': '#fadbd8'},
                              {'range': [target, gauge_max], 'color': '#d5f5e3'}],
                    'threshold': {'line': {'color': 'red', 'width': 3}, 'thickness': 0.8, 'value': target}
                }
            ), row=1, col=1)

            fig.add_trace(go.Bar(x=['Plan'], y=[rec['sheet_rev']], name='Sheet Revenue',
                marker_color='#3498db', text=f'${rec["sheet_rev"]/1e6:.2f}M', textposition='inside'), row=1, col=2)
            fig.add_trace(go.Bar(x=['Plan'], y=[rec['elem_rev']], name='Element Revenue',
                marker_color='#2ecc71', text=f'${rec["elem_rev"]/1e6:.2f}M', textposition='inside'), row=1, col=2)
            fig.add_trace(go.Bar(x=['Plan'], y=[-rec['cost']], name='Production Cost',
                marker_color='#e74c3c', text=f'-${rec["cost"]/1e6:.2f}M', textposition='inside'), row=1, col=2)

            fig.update_layout(
                barmode='relative', height=360,
                margin=dict(l=30, r=30, t=50, b=30),
                legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
                title_text=f'Gross Margin: ${rec["margin"]:,.0f} ({rec["margin_pct"]:.1f}%)')

            try:
                fig.show(config={'responsive': True})
            except Exception:
                display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False,
                                         config={'responsive': True})))


w_go.on_click(on_analyze)

# ─── ASSEMBLE DASHBOARD ───
_card = lambda children, m='0 0 12px 0': widgets.VBox(children, layout=widgets.Layout(
    width='100%', border='1px solid #d9d9d9', padding='14px', margin=m))

inputs_card = _card([
    widgets.HTML(
        '<h3 style="margin:0 0 6px;">Gross Revenue Optimizer</h3>'
        '<p style="margin:0 0 10px;color:#555;font-size:0.9em;">'
        'Constants: 312 operating days/yr &middot; 1,250 m²/day/shift &middot; '
        '54 m²/element &middot; 172,000 m² committed sheet &middot; '
        'Element price = 1.83&times; sheet rate</p>'),
    widgets.HBox([w_target, w_cost, w_price],
                 layout=widgets.Layout(flex_flow='row wrap', gap='10px')),
    widgets.HBox([w_go], layout=widgets.Layout(margin='14px 0 0'))
])

results_card = _card([
    widgets.HTML('<h3 style="margin:0 0 6px;">Results</h3>'
                 '<p style="color:#888;margin:0;">Click <b>Analyze Plan</b> above.</p>'),
    w_out
], m='0')

display(widgets.VBox([inputs_card, results_card], layout=widgets.Layout(max_width='1000px')))